Quick Code for Tutorial

Backtesting for Analytical VaR. Cornish-Fisher VaR

Experimental Python

# Introduction

<li> This Python notebooks implements backtesting for Analytical VaR -- test for the number of breaches

<li> Please note other implementations of VaR backtesting and any exam tasks, are likely to have variation. For example, VaR formula can include an adjustment for  
μ
−
1
2
σ
T
  even if  
μ
=
0
 , and rolling window for  
σ
  computation might change.

<li>  There might be alternative requirements, such as EWMA volatility forecast  
σ
t
|
t
−
1
  might be requested prior to computing VaR based on it. Or a task might have additional requirements, such as computation and analysis of consequitive breaches.

<strong>Firm-wide risk scenario:</strong>

Imagine each day you calculate 99%/10D VaR from the available prior data, and perform its backtest. 99% is confidence level,  
c
  and so  
1
−
c
=
0.01
 . 10D stands for 10-day period.

Compute (a) the percentage of VaR breaches and (b) probability of breach in VaR, given a breach was observed for the previous period.

The purpose of backtest is to check if the number of VaR breaches is close to the theoretical 1%. That allows for 2.52 breaches (statistically speaking) within a trading year.

Despite the simplicity of backtesting design, the aggregated firm-wide risk typically evaluated by experienced risk managers.

<strong>Rolling Window</strong>

10-Day Value at Risk is computed on the rolling basis -- that is we use a rolling window of 21-42 observations (log-returns) in order to compute the standard deviation.

We don't want the rolling window to be too short (eg to compute std dev from 10 or 15 returns), but equally do not wish to compute from 60-100 returns which will make our Value at Risk too insensitive to recent volatility.

Please remember that regardless of the number of observations, the timescale of standard deviation is always DAILY. In order to compute 10-day VaR, we scalee

$$\sigma_{10D} = \sqrt{\sigma^2_{1D}+\sigma^2_{1D}+\ldots}=\sigma_{1D}\sqrt{10}$$


With Factor (Normal Factor) being a percentile of the Normal Distribution that `cuts' 1% on the tail, we here use crudely simplified formula,

$$VaR = Factor \times \sigma_{10D}$$

<strong>Data Loss</strong>

Day 1 index level needed to compute the first 1D log-return.

21 log-returns (observations) are needed to compute the first std dev, which would be of DAILY timescale.

For the last ten days of the dataset, we will have 10D VaR but no information about the forward return.

The above makes the Number of eligible comparsons between 10D VaR and 10-day forward return to be less then the number of returns.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.optimize import fsolve # this is required for conversion of empirical skewness and kurtosis into S,K parameters for CF Approximation

In [2]:
# We summarise all parameters

rolling_window = 21  # here is 21 observations

c = 0.99
inverse_cdf = norm.ppf(1-c)

factor = inverse_cdf 

print(f'Exact value of ICDF of the normal distribution for p = {1-c} is: \033[1m{inverse_cdf}\033[0m')

Exact value of ICDF of the normal distribution for p = 0.010000000000000009 is: -2.3263478740408408


Load S&P 500 index levels, compute log-returns and std dev
We refer to index values as levels as is customary in econometrics.

We also compute daily log-returns.

To remind, the sample standard deviation is computed on rolling basis, and the window is shifted by 1 observation (return).

In [3]:
# Read the Excel file
data = pd.read_excel('SP500_VaR.xlsx', index_col='Date')

# Ensure the 'Date' column is parsed as datetime type if it's not automatically recognized
data.index = pd.to_datetime(data.index)

# Verify the DataFrame structure
# print(data.head())

FileNotFoundError: [Errno 2] No such file or directory: 'SP500_VaR.xlsx'

In [ ]:
data['LogReturn'] = np.log(data['S&P500']) - np.log(data['S&P500'].shift(1))
#xl['Mean']  = xl['LogReturn'].rolling(21, min_periods=21).mean() 

data['STD']  = data['LogReturn'].rolling(rolling_window, min_periods=rolling_window).std()

In [ ]:
data['VAR10D'] =  factor * (data['STD']) * np.sqrt(10)

# Re-factored optimised code would do below -- or even precompute factor * np.sqrt(10), even though time scaling relates to STD DEV
# data['VAR10D'] =  factor * np.sqrt(10) * (data['STD']) 


data['Ret10D'] = np.log(data['S&P500'].shift(-10) / data['S&P500'] )

# .shift(-10) brings up the future price (index level), which will realise in 10 days

data['Breach'] = data['Ret10D'] <  data['VAR10D']

## Breaches in VaR
Compare that 10D VaR number computed 'in the past' -- notionally ten days ago -- to the realised return and check if your prediction about the worst loss was breached. Hence, the backtesting.

$$r_{10D,t}<VaR_{t-10}$$
 
means breach, given both numbers are negative:

<li> VaR is fixed at time t and compared to the return realised from  $t$ to $t+10$. A breach occurs when a realised 10-day return  $r_{10D,t}=\ln(S_{t+10}/S_t)$ is below.
<li> Nbreaches divided by Ncomparisons gives the percentage of breaches. Identify the eligible number of observations (VaR values) and the number of breaches.

In [ ]:
print(data.to_string())
#print(xl.to_string(columns=[0,1,7,9,10]))

In [ ]:
# Filter for the eligible Number of comparisons
data_eligible = data[pd.notnull(data['Ret10D']) & pd.notnull(data['VAR10D'])]

# Filter for breaches
data_breach = data[data['Breach'] == True]

In [ ]:
# Now works out the implied probability of a breach

N_breaches = data_breach.count()['Breach']
N_obs = data_eligible.count()['VAR10D']
Breaches = N_breaches/N_obs * 100

print("Nobs\tNbreaches\tPercentage")

print("{0:.4f}".format(N_obs),"\t", "{0:.4f}".format(N_breaches), "\t", "{0:.4f}".format(Breaches), "\t")

In [ ]:
print(data_breach)

In [ ]:
plt.title('Backtesting of Analytical VaR');
fig_size = plt.rcParams["figure.figsize"]
fig_size[0] = 20
fig_size[1] = 5
plt.rcParams["figure.figsize"] = fig_size
#fig, ax = plt.subplots()
#ax.set_xlim(0, 1009)

varPlt, = plt.plot(data.index, data['VAR10D'], color='RED');
r10dplt, = plt.plot(data.index, data['Ret10D']);

varBreachPlt = plt.scatter(data_breach.index, data_breach['VAR10D'], color='BLACK', marker='x');

plt.legend(["VaR 99%/10D","return_10D fwd","Breach"]);
plt.grid();

fig = plt.figure();
ax1 = fig.add_subplot(211);
ax1.set_title('Index Level');
ax1.plot(data.index, data['S&P500'])
ax1.legend(["Index Level"]);
ax1.grid();
plt.show();

VaR_t plot starts after initial period -- as is expected.

data_breach.index keeps indexes (line numbers) the same as in original sample.

Alternative plot can be done using xl_eligible.index -- it will start about Day 21 position, where the VaR_t starts

-----------------

------------------

Cornish - Fisher Approximation for Empirical Percentile
In the Cornish-Fisher expansion, Skeweness  $S$ and Kurtosis   $K$
  are parameters that compute the percentile for a non-Normal random variable, which S&P 500 returns are empirically.

The relationship between Skeweness, Kurtosis parameters and the sample skewness   $\gamma_1$

  and the sample excess kurtosis  
$\gamma_2$  of the historic log-returns is not of transform. Instead,  
$\gamma_1$  and  
$\gamma_2$  each is function of  
$S$ , $K$ . The system of two equations has to be solved by numerical optimisation. It is described in full in the Anatomy of Cornish-Fisher note.

The above dillema has an easy solution: small values of actual skewness and kurtosis ( 
$\gamma_1$
,
$\gamma_2$
 ) roughly correspond to Skewness and Kurtosis parameters (S, K). However, there is an issue of validity domain: while S, K are parameters they have to have sensible values, eg sample skewness is not likely to be more than 2, and Skewness parameter is not likely to be much above 0.5-0.7.

In [ ]:
skewness = data['LogReturn'].skew()

# Pandas kurtosis method computes excess kurtosis, so no adjustment is needed
excess_kurtosis = data['LogReturn'].kurtosis()

print(f'Sample (actual) skewness of log-returns: {skewness}')
print(f'Sample (actual) excess kurtosis of log-returns: {excess_kurtosis}')

In [ ]:
def cornish_fisher_percentile(p, skewness, excess_kurtosis):
    """
    Compute the Cornish-Fisher percentile based on skewness and excess kurtosis.
    
    - p: the probability for which to compute the percentile.
    - skewness
    - excess_kurtosis
    
    Returns:
    - The adjusted z-score (percentile) based on the Cornish-Fisher expansion.
    """
    # Standard normal quantile (z-score for the given probability)
    z = norm.ppf(p)
    
    # Cornish-Fisher expansion
    z_adj = z + (1/6)*(z**2 - 1)*skewness + (1/24)*(z**3 - 3*z)*excess_kurtosis - (1/36)*(2*z**3 - 5*z)*skewness**2
    
    return z_adj

In [ ]:
adjusted_percentile = cornish_fisher_percentile(1-c, skewness, excess_kurtosis)

print(f'Adjusted percentile (z-score) for 1-c = {1-c}: {adjusted_percentile}')

In [ ]:
data['CF_VAR10D'] =  -0.5 * (data['STD'])**2 * 10 + adjusted_percentile * (data['STD']) * np.sqrt(10)

data['CF_Breach'] = data['Ret10D'] <  data['CF_VAR10D']

data_breach_cf = data[data['CF_Breach'] == True]

In [ ]:
# Filter for the eligible Number of comparisons
data_eligible = data[pd.notnull(data['Ret10D']) & pd.notnull(data['VAR10D'])]

# Filter for breaches
data_breach = data[data['Breach'] == True]

In [ ]:
plt.title('Backtesting of Cornish-Fisher VaR');
fig_size = plt.rcParams["figure.figsize"]
fig_size[0] = 20
fig_size[1] = 5
plt.rcParams["figure.figsize"] = fig_size
#fig, ax = plt.subplots()
#ax.set_xlim(0, 1009)

varPlt, = plt.plot(data.index, data['CF_VAR10D'], color='RED');
r10dplt, = plt.plot(data.index, data['Ret10D']);

varBreachPlt = plt.scatter(data_breach_cf.index, data_breach_cf['CF_Breach'], color='BLACK', marker='x');

plt.legend(["VaR 99%/10D","return_10D fwd","Breach"]);
plt.grid();

fig = plt.figure();
ax1 = fig.add_subplot(211);
ax1.set_title('Index Level');
ax1.plot(data.index, data['S&P500'])
ax1.legend(["Index Level"]);
ax1.grid();
plt.show();

Explore Skewness and Excess Kurtosis

In [ ]:
rolling_window = 42
shift_period = 5

# Compute rolling skewness
xl['RollingSkewness'] = xl['LogReturn'].rolling(window=rolling_window, min_periods=rolling_window).skew()

# Compute rolling kurtosis
xl['RollingKurtosis'] = xl['LogReturn'].rolling(window=rolling_window, min_periods=rolling_window).kurt()

In [ ]:
# For plotting every 5th day from the rolling calculations
selected_skewness = xl['RollingSkewness'][::shift_period]
selected_kurtosis = xl['RollingKurtosis'][::shift_period]

In [ ]:
# Plotting
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(selected_skewness.index, selected_skewness)
plt.title('Rolling Skewness (rolling window, shift period -- per code above)')
plt.xlabel('Historic Obs')
plt.ylabel('Skewness')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(selected_kurtosis.index, selected_kurtosis)
plt.title('Rolling Kurtosis (rolling window, shift period -- per code above)')
plt.xlabel('Historic Obs')
plt.ylabel('Kurtosis')
plt.legend()

plt.tight_layout()
plt.show()